# SO101 连续控制 · v3 DDPG（state 观测，确定性策略梯度）

**这个 notebook 在做什么**：在 SO101 ReachCube 任务上，用只含关节 + 物体位姿的**状态观测**，
从零训一个 DDPG 策略——确定性 Actor 直接输出连续动作，标量 Critic 学 Q(s,a)，配经验回放 +
目标网络软更新，动作上叠高斯噪声做探索。它是 SO101 连续控制阶梯的第一级。

**为什么从 DQN 升级到 DDPG**：上一包 `3_1` 的 DQN 靠对 Q 值取 argmax 选动作，动作一旦连续
（机械臂的关节力矩）就没法穷举了。DDPG 换成"确定性策略直接吐动作 + 沿着 Q 的梯度往上爬"，
这是从离散动作迈进连续动作的关键一步；经验回放和目标网络这两根稳定支柱从 DQN 原样复用，
只是目标网络这里既滞后化 critic 也滞后化 actor。

**观测归一化是这一级能学起来的前提**：SO101 的 55 维 state 里混着关节角、速度、物体位姿
等尺度完全不同的量，某些维度原始值能到 ~200。直接喂进 `DeterministicActor` 末层的 `Tanh`，
输入一大，tanh 早早饱和到 ±1 附近、梯度趋于 0——actor 的权重根本更新不动，不是算法学不动，
是网络输入没做基本预处理就先把自己饱和死了。修法是 `RunningNorm`：在线维护 state 每一维的
running mean/var（采集时用原始 state 更新统计），真正喂进网络前先归一化成零均值单位方差，
回放池里仍然存原始 state。

**诚实的教学结论（公平预算 500 iter、三级同预算、加观测归一化后的真实结果）**：加了归
一化后 actor **确实解冻了**——权重不再是一动不动，奖励也不再是死平的一条直线；但把预算
从 200 轮拉到 500 轮，"不稳"并没有随时间自愈：曲线全程在正负之间反复起落，500 轮下来
mean_reward（靠近度）均值只有约 0.28，success_once 从头到尾贴着 0，一次没跨过。这正是
"确定性策略 + 一份容易高估的单 Q + 只靠固定高斯噪声探索"在操作任务上的真实弱点：Q 一旦被
高估，确定性策略梯度就把 actor 整个推向那个虚高的动作，reward 应声跳水，直到 Q 估计慢慢
被纠正回来，进入下一轮同样的震荡——不是学不动，是学得不稳，而且始终探索不到稀疏成功判定
需要的精确位置，给多少轮预算都一样。放进三级同预算的阶梯里看：mean_reward（靠近度）是从
这一级的 0.28 起步的最低点，success_once 也是三级里唯一"从未离开过 0"的一档。缺什么：
整条链路只有一个 Q，容易被过高估计——下一级 TD3 用双 Q 取 min（Clipped Double Q）先把
"稳"找回来，看能不能借此摸到成功。

**四件套骨架**：`DeterministicActor`/`QCritic`（`nn.Module`）→ `DDPG`（`LightningModule`，
`automatic_optimization=False` 手动优化 actor / critic）→ `ReplayBuffer` + `SO101DDPGData`
（在线采集的 `LightningDataModule`）→ `trainer.fit(model, data)`。`RunningNorm` 单独一份，
在采集时更新统计、在喂进网络前做归一化。

> **运行方式**：SO101 仿真需要 GPU（ManiSkill GPU 后端）。自上而下逐 cell 运行；
> 打印格式与本模块 v4/v5 一致（`iter N: success_once=X  mean_reward=Y`），可直接对照三条曲线。
> 训好的 actor + 归一化统计存到 `DATASETS_ROOT/models/trained/so101_sim_offpolicy/<task>/ddpg.pt`。

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torch.utils.data import DataLoader, IterableDataset

import so101_sim  # 统一环境：lerobot 评测与 RL 训练共用同一份定义（editable 安装，直接 import）

## 0 RunningNorm：在线观测归一化

SO101 的 state 里关节角、速度、物体位姿混杂在一起，某些维度原始值能到 ~200；`DeterministicActor`
末层是 `Tanh`，大数值直接喂进去会把它饱和到 ±1 附近、梯度趋于 0——actor 的权重根本更新不动。
`RunningNorm` 用 Welford 并行算法在线维护每一维的 running mean/var：采集时用原始 state 更新
统计，真正喂进网络前再 `normalize` 成零均值单位方差。回放池里仍然存原始 state，不受影响。

In [ ]:
class RunningNorm(nn.Module):
    """在线观测归一化：跟踪 state 每一维的 running mean/std，把原始观测（值域可到 ~200）
    归一化到零均值单位方差，避免大数值让 actor 的 tanh 饱和、梯度冻结。"""

    def __init__(self, dim):
        super().__init__()
        self.register_buffer("mean", torch.zeros(dim))
        self.register_buffer("var", torch.ones(dim))
        self.register_buffer("count", torch.tensor(1e-4))

    @torch.no_grad()
    def update(self, x):
        bm, bv, bc = x.mean(0), x.var(0, unbiased=False), x.shape[0]
        delta = bm - self.mean; tot = self.count + bc
        self.mean += delta * bc / tot
        M2 = self.var * self.count + bv * bc + delta**2 * self.count * bc / tot
        self.var = M2 / tot; self.count = tot

    def normalize(self, x):
        return (x - self.mean) / (self.var.sqrt() + 1e-8)

## 1 DeterministicActor：state → 确定性动作

这一级的核心变化：Actor 不再像 REINFORCE/PPO 那样输出一个分布再采样，而是直接吐出一个
确定性动作。MLP 输出经 `tanh` 压到 [-1, 1]，再用动作区间线性缩放进 [low, high]
（`action_scale`/`action_bias` 注册成 buffer，随模型走 GPU）。确定性策略本身不探索——
探索靠后面 `sample_action` 叠的那点高斯噪声。

In [ ]:
class DeterministicActor(nn.Module):
    """state → 确定性动作（DPG，没有分布也不采样）：MLP 到 [-1,1] 再线性缩放进 [low, high]。"""

    def __init__(self, state_dim, action_dim, action_low, action_high):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, action_dim), nn.Tanh(),
        )
        self.register_buffer("action_scale", (action_high - action_low) / 2.0)
        self.register_buffer("action_bias", (action_high + action_low) / 2.0)

    def forward(self, state):
        return self.net(state) * self.action_scale + self.action_bias

## 2 QCritic：(state, action) → 标量 Q

和 DQN 逐动作打分的 Q 表不同，连续动作没法逐动作枚举，所以 Critic 把 state 和 action
**拼在一起**喂进 MLP，只输出一个标量 Q(s,a)。Actor 更新时就靠它给"当前会选的动作"打分、
反传出确定性策略梯度。

In [ ]:
class QCritic(nn.Module):
    """(state, action) → 标量 Q：拼接后过 MLP，只输出一个数值（不是分布，也不是逐动作打分表）。"""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, state, action):
        return self.net(torch.cat([state, action], dim=-1)).squeeze(-1)

## 3 DDPG：LightningModule，手动优化

持有 online / target 两套 actor + critic（初始化权重同步），外加一份 `obs_norm`（`RunningNorm`）。
`training_step` 是一个 minibatch 的完整 DDPG 更新，分三步：
1. **评论家**：目标 Q = `r + γ·Q_target(s', actor_target(s'))`（`no_grad`，下一步动作由**目标
   actor** 给出，避免自己追自己），critic 对它做 MSE 回归（always-bootstrap，对齐 v6 不用 done）；
2. **演员**：`-Q(s, actor(s)).mean()`，直接最大化评论家给"actor 当前会选的动作"打的分——这就是
   确定性策略梯度；
3. **软更新**：actor / critic 的目标网络各按 `τ` 缓慢跟随在线网络。

`state`/`next_state` 在喂进 actor/critic 前统一过 `self.obs_norm.normalize(...)`——buffer 里存
的还是原始 state，归一化只发生在"读出来喂网络"这一步。`sample_action` 给归一化后的确定性动作
叠高斯噪声再 clamp 回区间（训练期探索）；`eval_action` 无噪声。`automatic_optimization=False`，
backward / step 全手动。

In [ ]:
class DDPG(L.LightningModule):
    """一个 minibatch 的 DDPG 更新：评论家(MSE 回归 Q) → 演员(确定性策略梯度) → 软更新目标网络。"""

    def __init__(self, state_dim, action_dim, action_low, action_high,
                 gamma=0.99, tau=0.005, lr=3e-4, exploration_noise=0.1):
        super().__init__()
        self.automatic_optimization = False
        self.gamma, self.tau, self.lr = gamma, tau, lr
        self.exploration_noise = exploration_noise
        self.action_low, self.action_high = action_low, action_high

        self.actor = DeterministicActor(state_dim, action_dim, action_low, action_high)
        self.critic = QCritic(state_dim, action_dim)
        self.actor_target = DeterministicActor(state_dim, action_dim, action_low, action_high)
        self.critic_target = QCritic(state_dim, action_dim)
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.critic_target.load_state_dict(self.critic.state_dict())
        # 观测归一化：buffer 里存的仍是原始 state，这里只在喂进网络前做归一化
        self.obs_norm = RunningNorm(state_dim)

    def configure_optimizers(self):
        return (torch.optim.Adam(self.actor.parameters(), lr=self.lr),
                torch.optim.Adam(self.critic.parameters(), lr=self.lr))

    @torch.no_grad()
    def sample_action(self, state):
        """确定性动作 + 高斯探索噪声，clamp 回合法区间（确定性策略本身不探索，全靠这份噪声）。"""
        action = self.actor(self.obs_norm.normalize(state))
        noise = torch.randn_like(action) * self.exploration_noise
        return (action + noise).clamp(self.action_low, self.action_high)

    @torch.no_grad()
    def eval_action(self, state):
        return self.actor(self.obs_norm.normalize(state))

    def training_step(self, batch, batch_idx):
        actor_opt, critic_opt = self.optimizers()
        state, action, reward, next_state = batch
        # buffer 里是原始 state，喂进网络前统一归一化（统计量只在采集时更新，这里只读）
        state, next_state = self.obs_norm.normalize(state), self.obs_norm.normalize(next_state)

        # —— 评论家（MSE 回归 Q）：目标 Q 用目标网络算，动作来自目标 actor ——（always bootstrap）
        with torch.no_grad():
            next_action = self.actor_target(next_state)
            target_q = reward + self.gamma * self.critic_target(next_state, next_action)
        critic_loss = F.mse_loss(self.critic(state, action), target_q)
        critic_opt.zero_grad(); self.manual_backward(critic_loss); critic_opt.step()

        # —— 演员（确定性策略梯度）：直接最大化评论家给"actor 当前会选的动作"打的分 ——
        actor_loss = -self.critic(state, self.actor(state)).mean()
        actor_opt.zero_grad(); self.manual_backward(actor_loss); actor_opt.step()

        # —— 目标网络软更新（actor / critic 各一遍）——
        with torch.no_grad():
            for p, tp in zip(self.actor.parameters(), self.actor_target.parameters()):
                tp.mul_(1 - self.tau).add_(self.tau * p)
            for p, tp in zip(self.critic.parameters(), self.critic_target.parameters()):
                tp.mul_(1 - self.tau).add_(self.tau * p)

        self.log_dict({"critic_loss": critic_loss.detach(), "actor_loss": actor_loss.detach()},
                      prog_bar=True, on_step=True, on_epoch=False)

## 4 ReplayBuffer：经验回放（state 版）

定容环形缓冲区，整块开在 GPU 上；每步把所有并行环境的转移一次性滚动写入。state 版只存
state / action / reward / next_state 四样，没有 rgb——这是它比视觉 SAC 快得多的原因。

In [ ]:
class ReplayBuffer:
    """定容经验回放池，整块开在 GPU 上（state 版：无 rgb，只存关节状态向量）。"""

    def __init__(self, capacity, state_dim, action_dim, device):
        z = lambda *s: torch.zeros(*s, device=device)  # noqa: E731
        self.state = z(capacity, state_dim)
        self.next_state = z(capacity, state_dim)
        self.action = z(capacity, action_dim)
        self.reward = z(capacity)
        self.capacity, self.device = capacity, device
        self.pos, self.full = 0, False

    def __len__(self):
        return self.capacity if self.full else self.pos

    def add(self, state, action, reward, next_state):
        n = state.shape[0]
        idx = (torch.arange(n, device=self.device) + self.pos) % self.capacity
        self.state[idx] = state; self.next_state[idx] = next_state
        self.action[idx] = action; self.reward[idx] = reward.float()
        self.pos = (self.pos + n) % self.capacity
        self.full = self.full or self.pos < n

    def sample(self, batch_size):
        i = torch.randint(0, len(self), (batch_size,), device=self.device)
        return self.state[i], self.action[i], self.reward[i], self.next_state[i]

## 5 SO101DDPGData：在线采集的 DataModule

持有环境和回放池。`train_dataloader` 每个 epoch 重建一次（`reload_dataloaders_every_n_epochs=1`）：
先预热到 `learning_starts`（纯随机动作把池子灌起来），再用当前策略 `sample_action` 采
`steps_per_iter` 步入池、记下这轮的 `last_success` 和 `last_reward`（平均奖励，success 之外的
连续靠近度信号），最后从池里随机采 `updates_per_iter` 个 minibatch 逐个 yield 给 `training_step`。
`_collect` 里每次采到原始 state 都会先 `self.model.obs_norm.update(state)`——预热和正式采集都
更新，归一化统计跟着见过的所有 state 一起涨。这就是 off-policy"边玩边学、旧经验反复用"的数据流。

In [ ]:
class SO101DDPGData(L.LightningDataModule):
    """持有环境和回放池；每轮先采样、再把 minibatch 交给 Trainer（state 版，无 rgb）。

    `env` 是 ManiSkill 标准 `ManiSkillVectorEnv`：不像旧版 `TrainEnv` 那样自己缓存
    `self.obs`，这里改由本类持有 `self.state`，每次 `step` 后手动滚动到下一步。
    """

    def __init__(self, env, model, buffer, action_dim, steps_per_iter, updates_per_iter,
                batch_size, learning_starts):
        super().__init__()
        self.env, self.model, self.buffer = env, model, buffer
        self.action_dim = action_dim
        self.steps_per_iter, self.updates_per_iter = steps_per_iter, updates_per_iter
        self.batch_size, self.learning_starts = batch_size, learning_starts
        self.last_success = 0.0
        self.last_reward = 0.0
        self.state, _ = env.reset()

    def _collect(self, use_policy):
        state = self.state
        self.model.obs_norm.update(state)  # 用原始 state 更新归一化统计，每步一次
        if use_policy:
            action = self.model.sample_action(state)
        else:  # 预热：均匀随机动作把池子填起来
            low = torch.as_tensor(self.env.single_action_space.low, device=self.env.device)
            high = torch.as_tensor(self.env.single_action_space.high, device=self.env.device)
            action = low + (high - low) * torch.rand(self.env.num_envs, self.action_dim, device=self.env.device)
        next_state, reward, _, _, info = self.env.step(action)
        self.buffer.add(state, action, reward, next_state)
        self.state = next_state
        return info["success"].float().mean().item(), reward.float().mean().item()

    def train_dataloader(self):
        def gen():
            while len(self.buffer) < self.learning_starts:
                self._collect(use_policy=False)
            stats = [self._collect(use_policy=True) for _ in range(self.steps_per_iter)]
            succ, rew = zip(*stats)
            self.last_success = float(np.mean(succ))
            self.last_reward = float(np.mean(rew))
            for _ in range(self.updates_per_iter):
                yield self.buffer.sample(self.batch_size)

        class _DS(IterableDataset):
            def __iter__(self_inner):
                return gen()

        return DataLoader(_DS(), batch_size=None)

## 6 SuccessLogger：每轮打印成功率 + 平均奖励 + 定期存 ckpt

每个 epoch 末打印 `iter N: success_once=X  mean_reward=Y`（与本模块 v4/v5 完全同格式，三条曲线
可直接对照）：`success_once` 是硬指标（碰没碰到），`mean_reward` 是软指标（靠近了多少，success
长期为 0 时唯一能看出"学没学"的信号）。周期性把 actor 权重 **和** `obs_norm` 的归一化统计一起
存盘——评测/复用时离了归一化统计，喂进网络的还是没意义的原始尺度。

In [ ]:
class SuccessLogger(L.Callback):
    """每轮打印采集成功率 + 平均奖励（靠近度，success 之外的连续信号）+ 定期存 ckpt。"""

    def __init__(self, ckpt_dir, save_interval, max_iterations):
        self.ckpt_dir, self.save_interval, self.max_iterations = ckpt_dir, save_interval, max_iterations

    def on_train_epoch_end(self, trainer, pl_module):
        it = trainer.current_epoch + 1
        succ = trainer.datamodule.last_success
        rew = trainer.datamodule.last_reward
        print(f"  iter {it}: success_once={succ:.2f}  mean_reward={rew:.3f}", flush=True)
        if it % self.save_interval == 0 or it == self.max_iterations:
            self.ckpt_dir.mkdir(parents=True, exist_ok=True)
            torch.save({"actor": pl_module.actor.state_dict(),
                       "obs_norm": pl_module.obs_norm.state_dict()}, self.ckpt_dir / "ddpg.pt")

## 7 组装训练

四件套到位：环境 `so101_sim.state_rl_env(...)`（只给关节状态，跳过渲染管线）、
模型 `DDPG`、数据 `SO101DDPGData`、训练逻辑（手动优化的 DDPG 更新）。入口还是标准
Lightning 姿势 `trainer.fit(model, datamodule)`。下面 `run_training` 用的是与
v4 TD3 / v5 SAC **同一套预算**——num_envs=1024、UTD=256、批 512、回放池 50 万、
500 iter——好让 v3/v4/v5 只差算法、可公平对照。

In [ ]:
def run_training(task, num_envs, max_iterations, updates_per_iter, batch_size,
                 buffer_capacity, learning_starts, device, seed=1):
    torch.manual_seed(seed)
    env = so101_sim.state_rl_env(task, num_envs=num_envs)
    state_dim = env.single_observation_space.shape[-1]
    action_dim = env.single_action_space.shape[-1]
    low = torch.as_tensor(env.single_action_space.low, device=device)
    high = torch.as_tensor(env.single_action_space.high, device=device)
    model = DDPG(state_dim, action_dim, low, high).to(device)
    buffer = ReplayBuffer(buffer_capacity, state_dim, action_dim, device)
    data = SO101DDPGData(env, model, buffer, action_dim, steps_per_iter=1, updates_per_iter=updates_per_iter,
                        batch_size=batch_size, learning_starts=learning_starts)

    ckpt_dir = Path(os.environ["DATASETS_ROOT"]) / "models" / "trained" / "so101_sim_offpolicy" / task
    trainer = L.Trainer(
        accelerator="gpu", devices=1, max_epochs=max_iterations,
        reload_dataloaders_every_n_epochs=1, enable_checkpointing=False, logger=False,
        enable_model_summary=False, enable_progress_bar=False, log_every_n_steps=10,
        callbacks=[SuccessLogger(ckpt_dir, save_interval=25, max_iterations=max_iterations)],
    )
    trainer.fit(model, datamodule=data)
    env.close()
    return ckpt_dir / "ddpg.pt"

In [ ]:
# 改这里选任务与训练时长，然后 `python train_v3_ddpg.py`。
TASK = "SO101PickPlaceCube40-v1"

if __name__ == "__main__":
    # 与 v4 TD3 / v5 SAC 同一套共享预算，三级只差算法、可公平对照：
    # 每步 256 次更新（UTD），批 512，回放池 50 万；SAC 要到 iter~480 才稳定跨过"解决"，
    # 三级统一给够 500 iter 预算才公平——预算给短了，压根看不出 SAC 真正的优势在哪。
    run_training(
        task=TASK, num_envs=1024, max_iterations=500, updates_per_iter=256, batch_size=512,
        buffer_capacity=500_000, learning_starts=5_000, device="cuda",
    )